In [0]:
%pip install markdown python-docx beautifulsoup4

In [0]:
import markdown
from bs4 import BeautifulSoup
from docx import Document
from docx.shared import Inches
import requests
from io import BytesIO
import shutil

import re
import requests
import json

from io import BytesIO
from docx import Document
from docx.shared import Inches

In [0]:
#DOC_NAME = "agent_report_20260512_150123"
#DOC_NAME = "agent_report_20260513_132309"
DOC_NAME = "agent_report_20260514_075438"

REPORT_PATH = f"/Volumes/agentbricks/volumes/agent_reports/{DOC_NAME}.md"


DOCX_OUTPUT_PATH = (
    "/Volumes/agentbricks/volumes/agent_reports/"
    f"{DOC_NAME}.docx"
)

LOCAL_DOCX_OUTPUT_PATH = f"/tmp/{DOC_NAME}.docx"

VOLUME_DOCX_OUTPUT_PATH = (
    "/Volumes/agentbricks/volumes/agent_reports/"
    f"{DOC_NAME}.docx"
)

In [0]:
def extract_markdown_content(raw_text: str) -> str:
    text = raw_text.strip()

    # Try JSON parse first
    try:
        parsed = json.loads(text)

        if isinstance(parsed, list) and parsed:
            first_item = parsed[0]
            if isinstance(first_item, dict) and "content" in first_item:
                text = first_item["content"].strip()

        elif isinstance(parsed, dict) and "content" in parsed:
            text = parsed["content"].strip()

    except Exception:
        pass

    # Convert literal escaped newlines to real newlines
    text = text.replace("\\n", "\n")

    # Remove wrapper prefix up to the first markdown heading
    first_heading = re.search(r"(?m)^#\s+", text)
    if first_heading:
        text = text[first_heading.start():].strip()

    return text

In [0]:
import re
from io import BytesIO
from docx import Document
from docx.shared import Inches
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()


def markdown_to_docx(markdown_text: str, output_path: str):
    doc = Document()
    image_pattern = r"!\[(.*?)\]\((.*?)\)"

    for line in markdown_text.split("\n"):
        line = line.strip()

        if not line:
            continue

        image_match = re.search(image_pattern, line)

        if image_match:
            alt_text = image_match.group(1)
            image_path = image_match.group(2).strip()

            try:
                downloaded = w.files.download(file_path=image_path)
                image_bytes = downloaded.contents.read()

                doc.add_picture(
                    BytesIO(image_bytes),
                    width=Inches(6),
                )

                if alt_text:
                    doc.add_paragraph(alt_text)

            except Exception as e:
                doc.add_paragraph(f"[Image failed to load: {image_path}]")
                doc.add_paragraph(f"Error: {str(e)}")

            continue

        if line.startswith("# "):
            doc.add_heading(line.replace("# ", "", 1), level=1)

        elif line.startswith("## "):
            doc.add_heading(line.replace("## ", "", 1), level=2)

        elif line.startswith("### "):
            doc.add_heading(line.replace("### ", "", 1), level=3)

        elif line.startswith("- "):
            doc.add_paragraph(line.replace("- ", "", 1), style="List Bullet")

        elif re.match(r"^\d+\.", line):
            doc.add_paragraph(line, style="List Number")

        else:
            doc.add_paragraph(line)

    doc.save(output_path)

In [0]:
with open(REPORT_PATH, "r", encoding="utf-8") as f:
    raw_report = f.read()

clean_markdown = extract_markdown_content(raw_report)

html = markdown.markdown(
    clean_markdown,
    extensions=["tables", "fenced_code"]
)

displayHTML(html)

In [0]:
WORKSPACE_DOCX_OUTPUT_PATH = (
    "/Workspace/Users/ondrej.cerny@datasentics.com/test/reports_depot/"
    f"{DOC_NAME}.docx"
)

markdown_to_docx(
    markdown_text=clean_markdown,
    output_path=WORKSPACE_DOCX_OUTPUT_PATH,
)

print(f"DOCX saved to: {WORKSPACE_DOCX_OUTPUT_PATH}")
# shutil.copyfile(
#     LOCAL_DOCX_OUTPUT_PATH,
#     VOLUME_DOCX_OUTPUT_PATH,
# )

# print(f"DOCX saved to: {VOLUME_DOCX_OUTPUT_PATH}")